In [ ]:
import importlib
from auroral_processing import hdf_utils
from pathlib import Path
import os
from tqdm.auto import tqdm
import datetime
from datetime import timezone
import sys
import h5py
# sys.path.append('/Users/michaellavender/Documents/BUSPC/batch_processing')
from auroral_processing.binary_to_hdf import whole_binary_to_hdf

In [ ]:
importlib.reload(hdf_utils)

# Conversion

In [ ]:
from histutils.convert.__main__ import convert_DMC_to_hdf5
# importlib.reload(histutils.convert.__main__)

In [ ]:
# Looping
hst_drive = Path('/Volumes/HST13/')
dmc_files = list(hst_drive.rglob('*DMCdata'))

dmc_files.pop(4)
dmc_files.pop(0)


out_dir = Path('/Volumes/Elements/output727')

for f in dmc_files:
    print(f)
    out_fn = out_dir / f.with_suffix('.h5').name
    out_fn.parent.mkdir(parents=True, exist_ok=True)
    convert_DMC_to_hdf5(f, out_fn, {"header_bytes": 4})
    print('.' * 40 + '\n')

In [ ]:
dmcfn = Path('/Volumes/HST3/2013-04-06/2013-04-06T07-00-CamSer1387_frames533032-762470.DMCdata')
outfile = Path('/Volumes/Elements/output721/2013-04-06T07-00-CamSer1387_frames533032-762470.h5')
outfile.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# nmea and xml same name as hdf
convert_DMC_to_hdf5(dmcfn, outfile, {"header_bytes": 4})


In [ ]:
# nmea and xml different name as hdf
convert_DMC_to_hdf5(dmcfn, outfile, {"header_bytes": 4, 'nmeaFile': '/Volumes/HST3/2013-04-06/2013-04-06T07-00-CamSer1387.nmea', 'xmlFile': '/Volumes/HST3/2013-04-06/2013-04-06T07-00-CamSer1387.xml'})

In [ ]:
# nmea and xml same name as hdf - no time string
log_fn = '/Volumes/Elements 1/2014-02-28/2014-02-28T04-00-CamSer7196.log'
start_time = hdf_utils.parse_log_for_start_time(log_fn)
print(start_time)
convert_DMC_to_hdf5(dmcfn, outfile, {"header_bytes": 4, "startUTC": start_time})

# Rough Keogram

In [ ]:
from pprint import pprint

drive_path = Path('/Volumes/Elements')
output_folders = [p for p in drive_path.glob("*") if p.is_dir() and "output" in p.name.lower()]

for folder in output_folders:
    print(f'Entering folder: {str(folder)}')
    h5_files = [f for f in folder.glob('*.h5')]
    for f in h5_files:
        if str(f) == '/Volumes/Elements/output721/2013-03-31T07-00-CamSer1387.h5':
            continue
        print(f'processing file: {str(f)}')
        try:
            hdf_utils.make_keogram_rougher(hdf_path=f, out_dir=folder, bin_size_seconds=60, sample_interval=3600)
        except Exception:
            print(
            

    print('.'*30)

In [ ]:
hdf_fn = Path('/Volumes/Elements/hdf/2013-03-30/2014-03-30T10-46-CamSer7196.h5')
out_dir = Path('/Volumes/Elements/hdf/2013-03-30/')
hdf_utils.make_keogram_rougher(hdf_path=hdf_fn, out_dir=out_dir, bin_size_seconds=60, sample_interval=3600)

# Batch rough keograms

In [ ]:
import os
hdf_parent_folder = '/Volumes/TOSHIBA EXT/HDF'
skip_dirs = {"HDF Tests", "4tb_drive1"}
files = []

# Get files
for root, dirs, filenames in os.walk(hdf_parent_folder):
    dirs[:] = [d for d in dirs if d not in skip_dirs]
    for name in filenames:
        if name.endswith(".h5"):
            files.append(os.path.join(root, name))

files = [Path(f) for f in files]
files = [f for f in files if not f.with_suffix(".png").exists()]

In [ ]:
# Process
done = []
errors = []
for file in tqdm(files):
    print(file.name)
    out_dir = file.parent
    try:
        hdf_utils.make_keogram_rougher(hdf_path=file, out_dir=out_dir, bin_size_seconds=60, sample_interval=3600)
        done.append(file)
    except Exception as e:
        errors.append((file, e))
        print(f"  failed: {e}")

Errors: 

[(PosixPath('/Volumes/TOSHIBA EXT/HDF/2013-03-24/2013-03-24T05-55-CamSer7196.h5'),
  ValueError('frames per bin 1344120 must be less than sample interval 3600')),
 (PosixPath('/Volumes/TOSHIBA EXT/HDF/2013-04-20/7196/2013-04-20-CamSer7196_reduced_time.h5'),
  ValueError('frames per bin 3710 must be less than sample interval 3600'))]


# Hourly Videos & Keograms

In [ ]:
from auroral_processing.consumers import keogram_hourly as kch
from auroral_processing.consumers import keogram as keogram_consumer
from auroral_processing.consumers import stats as stats_consumer
from auroral_processing.consumers import video as video_consumer
importlib.reload(kch)
importlib.reload(keogram_consumer)
importlib.reload(stats_consumer)
importlib.reload(video_consumer)
importlib.reload(hdf_utils)

In [ ]:
# Setup
start = datetime.datetime(2013, 4, 14, 7, 40, 0, tzinfo=datetime.timezone.utc)
start_unix = start.timestamp() # needed for norm

hdf_fn = Path('/Volumes/TOSHIBA EXT/HDF/2013-04-20/7196/2013-04-20-CamSer7196.h5')
out_dir = Path('/Volumes/TOSHIBA EXT/Summary/2013-04-20/7196/outputs')
out_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
# Get norm
f = h5py.File(hdf_fn, 'r')
imgs = f['rawimg']
ut = f['ut1_unix']
# start_idx = hdf_utils.find_closest_item(ut, start_unix)
norm = hdf_utils.compute_norm(imgs, ut, 1, low_percentile=0.1, high_percentile=99.9)

In [ ]:
norm = hdf_utils.make_hourly_videos_keograms(hdf_path=hdf_fn, out_dir=out_dir, bin_size=5, playback_speed=15, output_hz=1, norm=norm)

### July 28

In [ ]:
h5_folders = [Path('/Volumes/Elements/output722'), Path('/Volumes/Elements/output723'), Path('/Volumes/Elements/output724')]
time = datetime.time(8, 0, 0, tzinfo=datetime.timezone.utc)

for folder in h5_folders:
    print(f'entering: {str(folder)}')
    h5_files = [f for f in folder.glob('*.h5')]

    for file in h5_files:
        print(f'processing: {str(file.name)}')
        try:
            date = get_date(file)
            dt = datetime.datetime.combine(date, time)
            norm = hdf_utils.compute_norm_from_hdf(hdf_fn=file, sample_interval_seconds=1, start_time=dt)
            out_dir = folder / 'processed' / file.stem
            out_dir.mkdir(parents=True, exist_ok=True)
            hdf_utils.make_hourly_videos_keograms(hdf_path=file, out_dir=out_dir, bin_size=5, playback_speed=15, output_hz=1, norm=norm)
        except Exception as e:
            print(f'file {str(file)} failed with error {e}')

In [ ]:
def get_date(h5_fn):
    date_str = Path(h5_fn).name.split("T")[0]
    date_obj = datetime.date.strptime(date_str, "%Y-%m-%d")
    return date_obj

## Aug 7

In [ ]:
h5_parent_folder = Path('/Volumes/Elements/DMC_output')
out_dir_parent = Path('/Users/labb/mlavender/DMC_videos_test')
out_dir_parent.mkdir(parents=True, exist_ok=True)
h5_files = [Path(f) for f in h5_parent_folder.iterdir() if f.suffix == '.h5']

for f in h5_files:
    print(f'processing: {str(f.name)}')
    try:
        out_dir = out_dir_parent / f.stem
        out_dir.mkdir(parents=True, exist_ok=True)
        hdf_utils.make_hourly_videos_keograms(hdf_path=f, out_dir=out_dir, bin_size=5, playback_speed=15, output_hz=1)
    except Exception as e:
        print(f'file {str(f)} failed with error {e}')

# Discrete video remaking

In [ ]:
out_dir1 = Path('/Volumes/Elements/HST/HST-videos-keograms/20130414/1387/renorm')
out_dir2 = Path('/Volumes/Elements/HST/HST-videos-keograms/20130414/7196/renorm')
h5_fn_1387 = Path('/Volumes/Elements/HST/data/HST-h5-organized/2013-04-14/1387/2013-04-14-CamSer1387.h5')
h5_fn_7196 = Path('/Volumes/Elements/HST/data/HST-h5-organized/2013-04-14/7196/2013-04-14-CamSer7196.h5')
start_time = datetime.datetime(2013, 4, 14, 8, 0, 0, tzinfo=timezone.utc)
end_time = datetime.datetime(2013, 4, 14, 8, 59, 59, tzinfo=timezone.utc)

hdf_utils.make_hourly_videos_keograms(hdf_path=h5_fn_1387, out_dir=out_dir1, start_time=start_time, end_time=end_time, bin_size=5, playback_speed=15, sample_rate_hz=1, make_keogram=False, camera='HST', serial=1387)
hdf_utils.make_hourly_videos_keograms(hdf_path=h5_fn_7196, out_dir=out_dir2, start_time=start_time, end_time=end_time, bin_size=5, playback_speed=15, sample_rate_hz=1, make_keogram=False, camera='HST', serial=7196)

# Helper Funcitions

In [ ]:
import datetime
import h5py
def get_time_range(hdf_fn):
    with h5py.File(hdf_fn, 'r') as f:
        ut = f['ut1_unix']
        start_epoch, end_epoch = ut[0], ut[-1]
        start = datetime.datetime.fromtimestamp(start_epoch, tz=datetime.timezone.utc)
        end = datetime.datetime.fromtimestamp(end_epoch, tz=datetime.timezone.utc)

    print(start.strftime("%Y-%m-%d %H:%M:%S"), end.strftime("%Y-%m-%d %H:%M:%S"))
    return start, end

In [ ]:
get_time_range(hdf_fn)